<div align="center">

# <font face="Gramond" color="teal"> Urdu Children’s Story Generation System </font>

<hr style="border: 0.1px solid white; width: 700px;">

</div>


## <font face="Garamond" color="mediumaquamarine"> Members </font>
 
- **Abdullah Attique** 
- **Minahil Rizwan** 
- **Talha Akram**


## <font face="Gramond" color="palevioletred"> All libraries </font>

In [1]:
from __future__ import annotations

import json
import random
import re
import time
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urlunparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm


c:\Users\pc planet\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### <font face="Gramond" color="peachpuff"> Phase I: Dataset Collection and Preprocessing </font>

In [8]:
# Phase I — Config + special tokens

BASE_TAG_URL = "https://www.rekhta.org/tags/children-s-story/children-s-stories?lang=ur"
BASE_ORIGIN = "https://www.rekhta.org"

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Special tokens as unused Unicode Private Use Area (PUA) characters.
# These are single characters (bytes/points) that are extremely unlikely to appear in Urdu text.
EOS = "\uE000"  # end of sentence
EOP = "\uE001"  # end of paragraph
EOT = "\uE002"  # end of story

SPECIAL_TOKENS = {
    "<EOS>": EOS,
    "<EOP>": EOP,
    "<EOT>": EOT,
}

@dataclass
class ScrapeConfig:
    min_stories: int = 200
    max_pages: int = 50  # safety cap; list page has 454 total stories (at time of writing)
    request_timeout_s: int = 30
    min_delay_s: float = 0.9
    max_delay_s: float = 1.8
    user_agent: str = (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0 Safari/537.36"
    )

CFG = ScrapeConfig()


def _sleep_polite(cfg: ScrapeConfig = CFG) -> None:
    time.sleep(random.uniform(cfg.min_delay_s, cfg.max_delay_s))


def make_session(cfg: ScrapeConfig = CFG) -> requests.Session:
    s = requests.Session()
    s.headers.update(
        {
            "User-Agent": cfg.user_agent,
            "Accept-Language": "ur,en-US;q=0.8,en;q=0.6",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        }
    )
    return s


def canonicalize_url(url: str) -> str:
    """Drop noisy query params while keeping lang=ur (important for Urdu pages)."""
    parsed = urlparse(url)
    q = parsed.query
    # Keep lang=ur only; drop sort etc.
    keep = "lang=ur" if "lang=ur" in q else ""
    return urlunparse((parsed.scheme, parsed.netloc, parsed.path, parsed.params, keep, parsed.fragment))


def is_paywall_or_limit_page(html: str) -> bool:
    # Rekhta shows a free-page/limit message for anonymous users.
    # If we detect it, we can skip this story and continue.
    return "free content pages" in html.lower() or "remaining out of" in html.lower()


print("Special tokens:", SPECIAL_TOKENS)
print("Output dir:", DATA_DIR.resolve())


Special tokens: {'<EOS>': '\ue000', '<EOP>': '\ue001', '<EOT>': '\ue002'}
Output dir: C:\Users\mtapl\Downloads\temp\data


In [ ]:
# Phase I — 1) Collect story links (until min. 200)

STORY_PATH_MARKER = "/children-s-stories/children-s-story/"


def fetch_soup(session: requests.Session, url: str, cfg: ScrapeConfig = CFG) -> BeautifulSoup:
    resp = session.get(url, timeout=cfg.request_timeout_s)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "lxml")


def extract_story_links_from_tag_page(soup: BeautifulSoup) -> List[str]:
    links: List[str] = []
    for a in soup.find_all("a", href=True):
        href = a.get("href")
        if not href:
            continue
        if STORY_PATH_MARKER not in href:
            continue
        abs_url = urljoin(BASE_ORIGIN, href)
        links.append(canonicalize_url(abs_url))

    # Deduplicate while preserving order
    seen: Set[str] = set()
    out: List[str] = []
    for u in links:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


_COLLECTION_LOADING_RE = re.compile(
    r"(/CollectionLoading\?[^\"']*id=children-s-story[^\"']*pageIndex=\d+[^\"']*)",
    flags=re.IGNORECASE,
)


def discover_collection_loading_template(session: requests.Session, cfg: ScrapeConfig = CFG) -> Optional[str]:
    """Discover Rekhta's JS pagination endpoint embedded in the tag page HTML."""
    resp = session.get(BASE_TAG_URL, timeout=cfg.request_timeout_s)
    resp.raise_for_status()
    html = resp.text
    m = _COLLECTION_LOADING_RE.search(html)
    if not m:
        return None
    path = m.group(1).replace("&amp;", "&")
    return urljoin(BASE_ORIGIN, path)


def build_collection_loading_url(template_url: str, page_index: int) -> str:
    return re.sub(r"pageIndex=\d+", f"pageIndex={page_index}", template_url)


def collect_story_links(session: requests.Session, cfg: ScrapeConfig = CFG) -> List[str]:
    # Rekhta uses an internal endpoint for pagination ("load more"):
    #   /CollectionLoading?...&pageIndex=N...
    # This reliably yields ~50 new items per pageIndex.
    session.headers.setdefault("Referer", BASE_TAG_URL)

    template = discover_collection_loading_template(session, cfg)
    all_links: List[str] = []
    seen: Set[str] = set()

    if template:
        print("Using CollectionLoading pagination endpoint.")
        for page_index in range(1, cfg.max_pages + 1):
            url = build_collection_loading_url(template, page_index)
            soup = fetch_soup(session, url, cfg)
            page_links = extract_story_links_from_tag_page(soup)

            new_count = 0
            for u in page_links:
                if u not in seen:
                    seen.add(u)
                    all_links.append(u)
                    new_count += 1

            print(f"PageIndex {page_index}: +{new_count} new links (total {len(all_links)})")
            if len(all_links) >= cfg.min_stories:
                break
            if new_count == 0 and page_index > 1:
                break
            _sleep_polite(cfg)

        return all_links

    # Fallback: just parse the first tag page if template discovery fails
    print("CollectionLoading endpoint not found; falling back to first-page parsing.")
    soup = fetch_soup(session, BASE_TAG_URL, cfg)
    return extract_story_links_from_tag_page(soup)


session = make_session(CFG)
story_links = collect_story_links(session, CFG)
print("Collected:", len(story_links))
print("Example link:", story_links[0] if story_links else None)


Using CollectionLoading pagination endpoint.
PageIndex 1: +50 new links (total 50)
PageIndex 2: +49 new links (total 99)
PageIndex 3: +50 new links (total 149)
PageIndex 4: +50 new links (total 199)
PageIndex 5: +50 new links (total 249)
Collected: 249
Example link: https://www.rekhta.org/children-s-stories/children-s-story/dadi-amma-ki-kahani-ishtiyaq-ahmad-children-s-stories?lang=ur


In [ ]:
# Phase I — 2) Story extraction + preprocessing + token injection

URDU_CHAR_RANGES = (
    "\u0600-\u06FF"  # Arabic
    "\u0750-\u077F"  # Arabic Supplement
    "\u08A0-\u08FF"  # Arabic Extended-A
    "\uFB50-\uFDFF"  # Arabic Presentation Forms-A
    "\uFE70-\uFEFF"  # Arabic Presentation Forms-B
)

# Allow Urdu/Arabic digits too
DIGIT_RANGES = "\u0660-\u0669\u06F0-\u06F9"

# Include common punctuation (Urdu + ASCII), whitespace, and our special tokens.
ALLOWED_PUNCT = "\n\t \r،۔؛؟!\"'“”‘’()\[\]{}:;,.…—-"
ALLOWED_SPECIAL = re.escape(EOS + EOP + EOT)

NON_URDU_RE = re.compile(
    rf"[^0-9{DIGIT_RANGES}{URDU_CHAR_RANGES}{re.escape(ALLOWED_PUNCT)}{ALLOWED_SPECIAL}]",
    flags=re.UNICODE,
)

MULTISPACE_RE = re.compile(r"[ \t\u00A0\u2000-\u200A]+")
MULTINEWLINE_RE = re.compile(r"\n{3,}")

# Basic Urdu character normalization (minimal, avoids over-normalizing)
URDU_CHAR_MAP = {
    "ي": "ی",  # Arabic Yeh -> Urdu Yeh
    "ك": "ک",  # Arabic Kaf -> Urdu Kaf
    "ھ": "ہ",  # do-chashmi heh -> heh (optional; helps standardize)
    "ۀ": "ہ",
    "ة": "ہ",
}


def normalize_unicode_urdu(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    for src, dst in URDU_CHAR_MAP.items():
        text = text.replace(src, dst)
    # Normalize punctuation variants
    text = text.replace("٫", ".").replace("٬", ",")
    text = text.replace("؟", "؟")
    return text


def strip_non_urdu(text: str) -> str:
    text = NON_URDU_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text)
    text = text.replace(" \n", "\n").replace("\n ", "\n")
    text = MULTINEWLINE_RE.sub("\n\n", text)
    return text.strip()


def standardize_punctuation(text: str) -> str:
    # Collapse repeated punctuation/spaces
    text = re.sub(r"[.]{3,}", "…", text)
    text = re.sub(r"[!]{2,}", "!", text)
    text = re.sub(r"[؟]{2,}", "؟", text)
    text = re.sub(r"[۔]{2,}", "۔", text)

    # Remove spaces before punctuation
    text = re.sub(r"\s+([،۔؛؟!,:;.])", r"\1", text)
    # Ensure a space after some punctuation (except newline)
    text = re.sub(r"([،؛:;,.!?])([^\s\n])", r"\1 \2", text)
    return text


def clean_story_text(raw_text: str) -> str:
    t = raw_text
    t = normalize_unicode_urdu(t)
    t = strip_non_urdu(t)
    t = standardize_punctuation(t)
    # Final whitespace cleanup
    t = re.sub(r"\n[ \t]+", "\n", t)
    t = MULTINEWLINE_RE.sub("\n\n", t)
    return t.strip()


# --- HTML extraction helpers ---

# These markers indicate the footer/extra sections that usually appear AFTER the main story text.
# DO NOT include "موضوعات" here; on Rekhta it appears near the top and would truncate the story.
STOP_MARKERS = [
    "مأخذ",  # source
    "ماخذ",
    "مزید دریافت",  # explore more
    "موضوعات",  # topics (appears both near top and footer)
    "Additional Links",
    "All rights reserved",
]


def extract_visible_text(soup: BeautifulSoup) -> str:
    # Remove non-content
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Prefer main/article containers that are likely to include the story body
    container = soup.find("main") or soup.find("article")
    if container is None:
        h1 = soup.find("h1")
        if h1 is not None:
            candidates = []
            for parent in h1.parents:
                if getattr(parent, "name", None) in {"main", "article", "section", "div"}:
                    candidates.append(parent)
                if len(candidates) >= 6:
                    break
            if candidates:
                container = max(candidates, key=lambda t: len(t.get_text(" ", strip=True)))

    if container is None:
        container = soup

    text = container.get_text("\n", strip=True)

    # Trim at stop markers (footer/extras)
    for m in STOP_MARKERS:
        # Some markers (like 'موضوعات') can appear in both header and footer.
        # For those, search only after a small offset so we trim the footer, not the header.
        if m == "موضوعات":
            idx = text.find(m, 500)
        else:
            idx = text.find(m)
        if idx != -1 and idx > 0:
            text = text[:idx]
            break

    return text


def parse_title_author(soup: BeautifulSoup) -> Tuple[Optional[str], Optional[str]]:
    title = soup.find("h1")
    title_text = title.get_text(strip=True) if title else None

    # On Rekhta story pages, author tends to be a link under an h2 near title.
    author_text = None
    if title is not None:
        # Look forward for the first author link
        h2 = title.find_next("h2")
        if h2 is not None:
            a = h2.find("a")
            if a is not None:
                author_text = a.get_text(strip=True)

    return title_text, author_text


def sentence_split_urdu(paragraph: str) -> List[str]:
    p = paragraph.strip()
    if not p:
        return []
    # Split on Urdu full stop, question mark, exclamation, and periods.
    # Keep delimiter by re-attaching.
    parts = re.split(r"([۔!?؟])", p)
    sents: List[str] = []
    buf = ""
    for part in parts:
        if not part:
            continue
        buf += part
        if part in {"۔", "!", "?", "؟"}:
            s = buf.strip()
            if s:
                sents.append(s)
            buf = ""
    tail = buf.strip()
    if tail:
        sents.append(tail)
    return sents


def add_special_tokens(clean_text: str) -> str:
    # Ensure the token stream is single-line per story (no embedded newlines).
    # Paragraphs are separated by blank lines; everything else becomes spaces.
    normalized = clean_text.replace("\r", "\n")
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", normalized) if p.strip()]
    out_paras: List[str] = []

    for para in paragraphs:
        # Collapse internal whitespace/newlines inside the paragraph before sentence splitting
        para = re.sub(r"\s+", " ", para).strip()
        sents = sentence_split_urdu(para)
        if not sents:
            continue
        # Append EOS to each sentence (and make sure each sentence is single-spaced)
        sent_stream = " ".join([re.sub(r"\s+", " ", s).strip() + EOS for s in sents])
        out_paras.append(sent_stream)

    story_stream = EOP.join(out_paras) + EOT
    # Extra safety: remove any stray newlines
    return re.sub(r"\s*\n\s*", " ", story_stream).strip()


# --- Scrape one story page ---

def scrape_one_story(session: requests.Session, url: str, cfg: ScrapeConfig = CFG) -> Dict[str, Optional[str]]:
    r = session.get(url, timeout=cfg.request_timeout_s)
    r.raise_for_status()
    html = r.text

    # IMPORTANT: Rekhta pages often include a "free content pages remaining" banner even when the full story text is present.
    # So we DO NOT early-return "blocked" just because that banner exists. We only mark blocked if we fail to extract text.
    soup = BeautifulSoup(html, "lxml")
    title, author = parse_title_author(soup)

    visible = extract_visible_text(soup)

    # Remove title/author lines if they occur in the visible text
    raw_text = visible
    if title:
        raw_text = raw_text.replace(title, " ")
    if author:
        raw_text = raw_text.replace(author, " ")

    raw_text = raw_text.strip()
    clean_text = clean_story_text(raw_text)
    token_text = add_special_tokens(clean_text) if clean_text else None

    if clean_text:
        status = "ok"
    else:
        status = "blocked" if is_paywall_or_limit_page(html) else "empty"

    return {
        "url": url,
        "title": title,
        "author": author,
        "raw_text": raw_text if raw_text else None,
        "clean_text": clean_text if clean_text else None,
        "token_text": token_text,
        "status": status,
    }

In [24]:
# Phase I — 2.5) Quick sanity-check (should show some OK stories)

# This tests a few pages end-to-end without running the full 200-story loop.
# It should print status=ok and non-zero clean_text lengths for at least some stories.

test_urls = story_links[:3] if "story_links" in globals() else []
print("Testing URLs:", len(test_urls))

for i, u in enumerate(test_urls, start=1):
    rec = scrape_one_story(session, u, CFG)
    clen = len(rec.get("clean_text") or "")
    tlen = len(rec.get("token_text") or "")
    print(f"{i}. status={rec.get('status')} clean_len={clen} token_len={tlen} title={rec.get('title')}")

Testing URLs: 3
1. status=ok clean_len=5670 token_len=5803 title=دادی اماں کی کہانی
2. status=ok clean_len=2623 token_len=2661 title=کر بھلا ہو بھلا
3. status=ok clean_len=5014 token_len=5109 title=کہاوتوں کی کہانیاں1


In [22]:
# Phase I — 3) Run full pipeline: scrape >=200, preprocess, save

# NOTE: Rekhta may restrict anonymous users (free content page limits). If many pages are "blocked",
# you may need to log in and provide cookies to `session`.

TARGET = 200

records: List[Dict[str, Optional[str]]] = []
blocked = 0
ok = 0

for url in tqdm(story_links, desc="Scraping stories"):
    rec = scrape_one_story(session, url, CFG)
    records.append(rec)

    if rec["status"] == "ok":
        ok += 1
    elif rec["status"] == "blocked":
        blocked += 1

    if ok >= TARGET:
        break

    _sleep_polite(CFG)

print(f"OK stories: {ok}")
print(f"Blocked/limited: {blocked}")
print(f"Total attempted: {len(records)}")

# Keep only successful stories with non-empty cleaned text
clean_records = [r for r in records if r.get("status") == "ok" and r.get("clean_text")]

expected_cols = ["url", "title", "author", "raw_text", "clean_text", "token_text", "status"]
df = pd.DataFrame(clean_records, columns=expected_cols)
print("Final dataset size:", len(df))

# Persist dataset
out_jsonl = DATA_DIR / "rekhta_children_stories_ur_clean.jsonl"
out_csv = DATA_DIR / "rekhta_children_stories_ur_clean.csv"
out_txt = DATA_DIR / "rekhta_children_stories_ur_token_stream.txt"

# jsonl (best for later processing)
df.to_json(out_jsonl, orient="records", lines=True, force_ascii=False)
# csv (handy to inspect metadata)
df.to_csv(out_csv, index=False)
# one-story-per-line token stream (useful for tokenizer)
if "token_text" in df.columns:
    token_lines = [re.sub(r"\s+", " ", t).strip() for t in df["token_text"].dropna().astype(str).tolist()]
else:
    token_lines = []
out_txt.write_text("\n".join(token_lines), encoding="utf-8")

print("Wrote:", out_jsonl)
print("Wrote:", out_csv)
print("Wrote:", out_txt)

# Sanity checks (do NOT print story content)
if len(df) > 0:
    lens = df["clean_text"].str.len()
    print("Clean text length — min/mean/max:", int(lens.min()), float(lens.mean()), int(lens.max()))
    print("Special token codepoints:", {k: hex(ord(v)) for k, v in SPECIAL_TOKENS.items()})

Scraping stories:  80%|███████▉  | 199/249 [07:59<02:00,  2.41s/it]


OK stories: 200
Blocked/limited: 0
Total attempted: 200
Final dataset size: 200
Wrote: data\rekhta_children_stories_ur_clean.jsonl
Wrote: data\rekhta_children_stories_ur_clean.csv
Wrote: data\rekhta_children_stories_ur_token_stream.txt
Clean text length — min/mean/max: 474 5641.225 39133
Special token codepoints: {'<EOS>': '0xe000', '<EOP>': '0xe001', '<EOT>': '0xe002'}


### <font face="Gramond" color="peachpuff"> Phase II: Tokenizer Training (BPE) </font>

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import re

# Config
DATA_PATH = Path("rekhta_children_stories_ur_token_stream.txt")
VOCAB_SIZE = 250
OUTPUT_DIR = Path("tokenizer")
OUTPUT_DIR.mkdir(exist_ok=True)

# Load Corpus

def load_corpus(path: Path):
    with open(path, encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
    return lines

# Initialize Vocabulary

def build_initial_vocab(lines):
    vocab = Counter()
    for line in lines:
        # Split into words by whitespace
        words = line.split()
        for word in words:
            # Represent word as list of characters
            chars = list(word)
            vocab[tuple(chars)] += 1
    return vocab


# Get Pair Frequencies
def get_pair_freq(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i+1])
            pairs[pair] += freq
    return pairs


# Merge Most Frequent Pair
def merge_pair(vocab, pair_to_merge):
    new_vocab = {}
    bigram = re.escape(' '.join(pair_to_merge))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')

    for word, freq in vocab.items():
        word_str = ' '.join(word)
        # Replace the pair with merged token
        merged_word = pattern.sub(''.join(pair_to_merge), word_str)
        new_word = tuple(merged_word.split())
        new_vocab[new_word] = freq

    return new_vocab


# Train BPE
def train_bpe(lines, vocab_size):
    vocab = build_initial_vocab(lines)

    merges = []
    current_vocab_size = len(set(symbol for word in vocab for symbol in word))

    print("Initial vocab size:", current_vocab_size)

    while current_vocab_size < vocab_size:
        pair_freq = get_pair_freq(vocab)
        if not pair_freq:
            break

        best_pair = pair_freq.most_common(1)[0][0]
        merges.append(best_pair)

        vocab = merge_pair(vocab, best_pair)

        current_vocab_size = len(set(symbol for word in vocab for symbol in word))

        print(f"Merged {best_pair} -> New vocab size: {current_vocab_size}")

    return merges, vocab


def save_tokenizer(merges, vocab):
    merges_path = OUTPUT_DIR / "bpe_merges.txt"
    vocab_path = OUTPUT_DIR / "bpe_vocab.txt"

    with open(merges_path, "w", encoding="utf-8") as f:
        for m in merges:
            f.write(f"{m[0]} {m[1]}\n")

    unique_tokens = sorted(set(symbol for word in vocab for symbol in word))

    with open(vocab_path, "w", encoding="utf-8") as f:
        for token in unique_tokens:
            f.write(token + "\n")

    print("Saved merges to:", merges_path)
    print("Saved vocab to:", vocab_path)


if __name__ == "__main__":
    lines = load_corpus(DATA_PATH)
    merges, final_vocab = train_bpe(lines, VOCAB_SIZE)
    save_tokenizer(merges, final_vocab)


Initial vocab size: 88
Merged ('۔', '\ue000') -> New vocab size: 88
Merged ('ہ', 'ی') -> New vocab size: 89
Merged ('ہ', 'ا') -> New vocab size: 90
Merged ('ن', 'ے') -> New vocab size: 91
Merged ('م', 'ی') -> New vocab size: 92
Merged ('ہ', 'و') -> New vocab size: 93
Merged ('ا', 'ن') -> New vocab size: 94
Merged ('ک', 'ی') -> New vocab size: 95
Merged ('ا', 'س') -> New vocab size: 96
Merged ('و', 'ر') -> New vocab size: 97
Merged ('ک', 'ر') -> New vocab size: 98
Merged ('ہ', 'ے') -> New vocab size: 99
Merged ('ک', 'ے') -> New vocab size: 100
Merged ('می', 'ں') -> New vocab size: 101
Merged ('ی', 'ا') -> New vocab size: 102
Merged ('س', 'ے') -> New vocab size: 103
Merged ('ک', 'ہ') -> New vocab size: 104
Merged ('ا', 'ور') -> New vocab size: 105
Merged ('ہی', 'ں') -> New vocab size: 106
Merged ('و', 'ں') -> New vocab size: 107
Merged ('ک', 'ا') -> New vocab size: 108
Merged ('ک', 'و') -> New vocab size: 109
Merged ('ئ', 'ی') -> New vocab size: 110
Merged ('د', 'ی') -> New vocab size: 1

In [ ]:
from pathlib import Path
import re

# Config
CORPUS_PATH = Path("rekhta_children_stories_ur_token_stream.txt")
MERGES_PATH = Path("tokenizer/bpe_merges.txt")
VOCAB_PATH = Path("tokenizer/bpe_vocab.txt")
OUTPUT_PATH = Path("tokenizer/bpe_tokenized_corpus.txt")

# Special token for End-of-Text (EOT)
EOT_TOKEN = "\uE002"

# BPE Tokenizer Class
class BPETokenizer:
    def __init__(self, merges_path):
        self.merges = self.load_merges(merges_path)

    def load_merges(self, path):
        merges = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                a, b = line.strip().split()
                merges.append((a, b))
        return merges

    def encode_word(self, word):
        """Encode a single word into BPE subwords"""
        tokens = list(word)
        while True:
            merged = False
            for a, b in self.merges:
                i = 0
                while i < len(tokens) - 1:
                    if tokens[i] == a and tokens[i+1] == b:
                        tokens[i:i+2] = [a+b]
                        merged = True
                    else:
                        i += 1
            if not merged:
                break
        return tokens

    def encode_line(self, line):
        """Encode a full line (story)"""
        words = line.strip().split()
        tokenized_words = []
        for word in words:
            tokenized_words.extend(self.encode_word(word))
        # Append EOT token
        tokenized_words.append(EOT_TOKEN)
        return tokenized_words

# Encode Entire Corpus
def encode_corpus():
    tokenizer = BPETokenizer(MERGES_PATH)

    with open(CORPUS_PATH, encoding="utf-8") as f_in, \
         open(OUTPUT_PATH, "w", encoding="utf-8") as f_out:
        for line in f_in:
            if not line.strip():
                continue
            tokens = tokenizer.encode_line(line)
            f_out.write(" ".join(tokens) + "\n")

    print("✅ Tokenized corpus saved to:", OUTPUT_PATH)

# Run
if __name__ == "__main__":
    encode_corpus()


✅ Tokenized corpus saved to: tokenizer\bpe_tokenized_corpus.txt


### <font face="Gramond" color="peachpuff"> Phase III: Trigram Language Model </font>

### <font face="Gramond" color="peachpuff"> Phase IV: Microservice & Containerization </font>

### <font face="Gramond" color="peachpuff"> Phase V: Web-based Frontend </font>

### <font face="Gramond" color="peachpuff"> Phase VI: Cloud Deployment </font>